# ALL ENGINES SHOWCASE: Google Colab GPU Render Pipeline

This notebook orchestrates the multi-engine rendering workflow combining:
- **Remotion** (React 19 + TypeScript Compositor with Chromium Angle-EGL)
- **HyperFrames** (DOM + GSAP 3 Seekable Motion System with GPU Video Encoding)
- **ManimGL** (3Blue1Brown Mathematical Engine with Headless OpenGL under Xvfb)
- **FFmpeg** (Stream validation and NVIDIA NVENC Hardware Transcode)

---
### Instructions
1. In the Colab menu, choose **Runtime -> Change runtime type** and ensure **T4 GPU** (or A100/L4) is selected.
2. Run the cells sequentially from top to bottom. Each stage is cached so rerunning cells avoids redundant re-renders.

## 1. Setup Workspace & Mount Google Drive
Mount Google Drive (for output export) and clone or locate the repository in `/content/ALL_ENGINES_SHOWCASE`.

In [ ]:
import os, sys, shutil, subprocess, zipfile
from pathlib import Path
from google.colab import drive, files

# Mount Google Drive
drive.mount('/content/drive', force_remount=False)

REPO_URL = "https://github.com/ela123/ALL_ENGINES_SHOWCASE.git"
PROJECT = Path("/content/ALL_ENGINES_SHOWCASE")

if not PROJECT.exists():
    print(f"Cloning {REPO_URL}...")
    res = subprocess.run(["git", "clone", REPO_URL, str(PROJECT)])
    if res.returncode != 0:
        print("Git clone failed. Prompting for manual ZIP upload fallback...")
        uploaded = files.upload()
        zip_name = next(iter(uploaded))
        with zipfile.ZipFile(zip_name) as z:
            z.extractall("/content")

print("Project root:", PROJECT, "| Exists:", PROJECT.exists())
assert PROJECT.exists(), "Project directory not found!"


## 2. Detect & Inspect Available GPU Hardware
Probes NVIDIA CUDA device status, driver version, and available compute.

In [ ]:
gpu_detected = False
try:
    nvidia_smi = subprocess.check_output(["nvidia-smi"], text=True)
    print(nvidia_smi)
    gpu_detected = True
except Exception as e:
    print("WARNING: No NVIDIA GPU detected. Ensure Runtime is set to GPU (Runtime -> Change runtime type -> GPU).")

print("GPU Available:", gpu_detected)


## 3. Install System Dependencies & Setup Node.js 22
Installs FFmpeg, Cairo, Pango, Mesa/OpenGL, Xvfb virtual framebuffer, and Node.js v22.

In [ ]:
# Marker to prevent redundant installations
SYS_MARKER = Path("/content/.sys_installed")
if not SYS_MARKER.exists():
    print("Installing system packages (FFmpeg, Pango, Cairo, OpenGL, Xvfb)...")
    !apt-get update -qq
    !apt-get install -y -qq ffmpeg libpango1.0-dev libcairo2-dev libgl1-mesa-dev libegl1-mesa-dev xvfb mesa-utils
    
    # Check Node.js version and install Node 22 if needed
    try:
        node_ver = subprocess.check_output(["node", "-v"], text=True).strip()
        major = int(node_ver.lstrip("v").split(".")[0])
    except Exception:
        major = 0
    if major < 22:
        print("Upgrading to Node.js 22...")
        !curl -fsSL https://deb.nodesource.com/setup_22.x | bash -
        !apt-get install -y -qq nodejs
    SYS_MARKER.touch()

print("Node version:", subprocess.check_output(["node", "-v"], text=True).strip())
print("NPM version:", subprocess.check_output(["npm", "-v"], text=True).strip())
print("FFmpeg version:", subprocess.check_output(["ffmpeg", "-version"], text=True).splitlines()[0])


## 4. Install Engine Dependencies
Installs HyperFrames (GSAP + CLI), ManimGL (Python package), and Remotion (v4.0.545 with React 19).

In [ ]:
# HyperFrames dependencies
hf_dir = PROJECT / "hyperframes"
if not (hf_dir / "node_modules").exists():
    print("Installing HyperFrames dependencies...")
    subprocess.run(["npm", "install"], cwd=hf_dir, check=True)
    subprocess.run(["npm", "install", "-g", "hyperframes"], check=True)

# ManimGL dependencies
manim_marker = Path("/content/.manim_installed")
if not manim_marker.exists():
    print("Installing ManimGL Python dependencies...")
    !pip install -q manimgl numpy
    manim_marker.touch()

# Remotion dependencies
remotion_dir = PROJECT / "remotion"
if not (remotion_dir / "node_modules").exists():
    print("Installing Remotion dependencies...")
    subprocess.run(["npm", "install"], cwd=remotion_dir, check=True)

print("All engine dependencies installed successfully.")


## 5. Render Configuration & Execution Options
Select whether you want a full production render or a fast preview, and whether to resume existing renders.

In [ ]:
# --- EXECUTION CONTROLS ---
RENDER_MODE = "full"        # Options: "full" (1080p high quality) | "preview" (faster test)
RESUME_EXISTING = True      # Skip scene if output file already exists
USE_GPU_ACCEL = True        # Request GPU hardware acceleration flags

RENDERS_DIR = PROJECT / "renders"
RENDERS_DIR.mkdir(parents=True, exist_ok=True)
(RENDERS_DIR / "manim").mkdir(parents=True, exist_ok=True)
PUBLIC_ASSETS = PROJECT / "remotion" / "public" / "assets"
PUBLIC_ASSETS.mkdir(parents=True, exist_ok=True)

# Ensure base assets are synced
base_assets = PROJECT / "assets"
for f in base_assets.rglob("*"):
    if f.is_file():
        target = PUBLIC_ASSETS / f.relative_to(base_assets)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, target)

print("Base assets synced to Remotion store. Ready to render.")


## 6. Render Specialist Scene: HyperFrames (HTML5 + CSS + GSAP 3)
Renders `hyperframes/index.html` at 1920x1080 @ 30 FPS into `renders/hyperframes-showcase.mp4`.

In [ ]:
hf_out = RENDERS_DIR / "hyperframes-showcase.mp4"
if RESUME_EXISTING and hf_out.exists():
    print(f"[SKIP] HyperFrames output already exists at {hf_out}")
else:
    print("Rendering HyperFrames scene...")
    quality_flag = "high" if RENDER_MODE == "full" else "low"
    cmd = ["npx", "hyperframes", "render", "--output", str(hf_out), "--fps", "30", "--quality", quality_flag]
    if USE_GPU_ACCEL:
        cmd.append("--gpu")
    try:
        subprocess.run(cmd, cwd=PROJECT / "hyperframes", check=True)
        print("Rendered HyperFrames with GPU acceleration.")
    except subprocess.CalledProcessError:
        print("GPU encoding unavailable; falling back to software encoding...")
        subprocess.run(["npx", "hyperframes", "render", "--output", str(hf_out), "--fps", "30", "--quality", quality_flag], cwd=PROJECT / "hyperframes", check=True)

assert hf_out.exists(), f"HyperFrames output failed to generate: {hf_out}"
shutil.copy2(hf_out, PUBLIC_ASSETS / "hyperframes-showcase.mp4")
print("HyperFrames render ready and synchronized.")


## 7. Render Specialist Scenes: ManimGL (2D & 3D Mathematical Animations)
Initializes virtual display `Xvfb :99` and renders `RiskAndCompounding` and `LeverageSurface`.

In [ ]:
os.environ["DISPLAY"] = ":99"
!Xvfb :99 -screen 0 1920x1080x24 > /dev/null 2>&1 &
import time; time.sleep(2)

manim_dir = RENDERS_DIR / "manim"
m2d_out = manim_dir / "manim-showcase.mp4"
m3d_out = manim_dir / "manim-3d-showcase.mp4"

# Render 2D Scene
if RESUME_EXISTING and m2d_out.exists():
    print(f"[SKIP] Manim 2D scene already exists at {m2d_out}")
else:
    print("Rendering ManimGL Scene 1: RiskAndCompounding...")
    subprocess.run([
        "manimgl", "scenes.py", "RiskAndCompounding",
        "-w", "-r", "1920x1080", "--fps", "30",
        "--video_dir", str(manim_dir), "--file_name", "manim-showcase"
    ], cwd=PROJECT / "manim", check=True)

# Render 3D Scene
if RESUME_EXISTING and m3d_out.exists():
    print(f"[SKIP] Manim 3D scene already exists at {m3d_out}")
else:
    print("Rendering ManimGL Scene 2: LeverageSurface (3D)...")
    try:
        subprocess.run([
            "manimgl", "scenes.py", "LeverageSurface",
            "-w", "-r", "1920x1080", "--fps", "30",
            "--video_dir", str(manim_dir), "--file_name", "manim-3d-showcase"
        ], cwd=PROJECT / "manim", check=True)
    except Exception as e:
        print("3D OpenGL scene encountered an issue; duplicating 2D render as fallback:", e)
        shutil.copy2(m2d_out, m3d_out)

assert m2d_out.exists(), f"ManimGL 2D render failed: {m2d_out}"
shutil.copy2(m2d_out, PUBLIC_ASSETS / "manim-showcase.mp4")
if m3d_out.exists():
    shutil.copy2(m3d_out, PUBLIC_ASSETS / "manim-3d-showcase.mp4")
print("ManimGL scenes ready and synchronized.")


## 8. Render Remotion Integrated Master Timeline
Composites all native sequences, transitions, audio tracks, and specialist renders into a single 60s (1800 frames) 1920x1080 @ 30 FPS master.

In [ ]:
master_out = RENDERS_DIR / "integrated-showcase.mp4"
if RESUME_EXISTING and master_out.exists():
    print(f"[SKIP] Integrated master already exists at {master_out}")
else:
    print("Rendering Remotion Integrated Showcase Master...")
    remotion_cmd = [
        "npx", "remotion", "render", "src/index.ts", "IntegratedShowcase",
        str(master_out), "--codec=h264", "--crf=18"
    ]
    # Try Angle-EGL GPU acceleration first on Linux
    try:
        subprocess.run(remotion_cmd + ["--gl=angle-egl"], cwd=PROJECT / "remotion", check=True)
        print("Remotion rendered with Angle-EGL GPU acceleration.")
    except subprocess.CalledProcessError:
        print("Angle-EGL unavailable; retrying with swangle GL fallback...")
        subprocess.run(remotion_cmd + ["--gl=swangle"], cwd=PROJECT / "remotion", check=True)

assert master_out.exists(), f"Remotion master render failed: {master_out}"
print("Master timeline successfully rendered:", master_out)


## 9. Validate Streams & GPU Transcode with FFmpeg NVENC
Verifies resolution, duration, and frame rate with `ffprobe`, then uses `h264_nvenc` to produce the final optimized distribution file.

In [ ]:
master_out = RENDERS_DIR / "integrated-showcase.mp4"
probe = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration:stream=width,height,r_frame_rate", "-of", "json", str(master_out)],
    capture_output=True, text=True, check=True
)
print("=== FFPROBE MASTER METADATA ===")
print(probe.stdout)

final_gpu = RENDERS_DIR / "final-showcase-gpu.mp4"
encoders = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True).stdout

if "h264_nvenc" in encoders:
    print("Encoding final distribution master with NVIDIA NVENC (h264_nvenc)...")
    subprocess.run([
        "ffmpeg", "-y", "-i", str(master_out),
        "-c:v", "h264_nvenc", "-preset", "p5", "-cq", "20",
        "-c:a", "aac", "-b:a", "192k", "-movflags", "+faststart",
        str(final_gpu)
    ], check=True)
    print("NVENC GPU transcode successful:", final_gpu)
else:
    print("NVENC encoder not present in FFmpeg; copying master to final distribution file.")
    shutil.copy2(master_out, final_gpu)

print("Final validated file:", final_gpu)


## 10. Export Videos to Google Drive
Copies all individual scene renders and the final integrated master video to your Google Drive folder: `MyDrive/ALL_ENGINES_SHOWCASE_OUTPUT`.

In [ ]:
DRIVE_DEST = Path("/content/drive/MyDrive/ALL_ENGINES_SHOWCASE_OUTPUT")
DRIVE_DEST.mkdir(parents=True, exist_ok=True)

rendered_files = list(RENDERS_DIR.rglob("*.mp4"))
for f in rendered_files:
    dest_file = DRIVE_DEST / f.name
    shutil.copy2(f, dest_file)
    print(f"Exported: {f.name} -> {dest_file} ({f.stat().st_size / (1024*1024):.2f} MB)")

print(f"\nAll {len(rendered_files)} videos successfully saved to Google Drive!")
